# Function Tool

Custom code that agents can call during conversations

In [1]:
import os
import inspect
import typing
import types
import pprint
from agent_framework import (
    tool,
    FunctionTool,
    FunctionInvocationContext,
    Agent
)
from agent_framework.openai import OpenAIChatClient
from typing import Never, Annotated, TypedDict
from pydantic import Field, BaseModel
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Note: if you don't specify the name and description parameters in the @tool decorator, the framework will automatically use the function's name and docstring as fallbacks.
@tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")], # required
    ctx: FunctionInvocationContext, # automatically detected by framework and marked as not required
    city: Annotated[str, Field(description="The city to get the weather for.")] = "Halifax", # not required
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [3]:
chat_client = OpenAIChatClient(
    base_url=os.getenv("FOUNDRY_OPENAI_BASE_URL"),
    api_key=os.getenv("FOUNDRY_API_KEY"),
    model=os.getenv("FOUNDRY_MODEL"),
)

agent = Agent(
    client=chat_client,
    name="Tool agent",
    description="You are a tool agent that can utilize various tools to assist with tasks.",
    instructions="You have access to various tools and should use them to assist with tasks effectively.",
    tools=[get_weather],
    context_providers=[],
)

## Inspect Function Tool

In [4]:
from typing import TypedDict

class ToolParameterInfo(TypedDict):
    name: str
    type: list[type]
    required: bool
    default: object | None

class ToolInfo(TypedDict):
    name: str
    description: str
    approval_mode: str
    parameters: list[ToolParameterInfo]
    return_type: list[type]

In [5]:
def is_function_tool(func) -> bool:
    return isinstance(func, FunctionTool)

def _extract_types(tp) -> list[type]:
    if tp is None:
        return []
    
    origin = typing.get_origin(tp)
    if origin in (typing.Union, types.UnionType):
        return [t for t in typing.get_args(tp)]
    return [tp]

def get_functiontool_return_type(tool: FunctionTool) -> list[type]:
    if not is_function_tool(tool):
        raise ValueError("Provided function is not a FunctionTool.")
    
    hints = typing.get_type_hints(tool.func)
    return _extract_types(hints.get("return", None))


def get_functiontool_parameters(tool: FunctionTool) -> list[ToolParameterInfo]:
    if not is_function_tool(tool):
        raise ValueError("Provided function is not a FunctionTool")

    if not tool.func:
        raise ValueError("FunctionTool has no associated function.")
    
    parameters = tool.parameters()
    properties = parameters.get("properties", None)
    if properties is None:
        return []

    required = parameters.get("required", [])

    sig = inspect.signature(tool.func)
    hints = typing.get_type_hints(tool.func)
    parameters_info = []
    
    for param_name, param in sig.parameters.items():
        param_type = hints.get(param_name, None)
        param_info = ToolParameterInfo(
            name=param_name,
            type=_extract_types(param_type),
            required=param_name in required,
            default=None if param.default is inspect.Parameter.empty else param.default
        )
        parameters_info.append(param_info)
    
    return parameters_info

def get_functiontool_metadata(tool: FunctionTool) -> ToolInfo:
    if not is_function_tool(tool):
        raise ValueError("Input must be an instance of FunctionTool")
    info: ToolInfo = {
        "name": tool.name,
        "description": tool.description,
        "approval_mode": tool.approval_mode,
        "parameters": get_functiontool_parameters(tool),
        "return_type": get_functiontool_return_type(tool),
    }
    return info


In [6]:
for tool in agent.default_options["tools"]:
    if is_function_tool(tool):
        pprint.pp(get_functiontool_metadata(tool), indent=4)

{   'name': 'get_weather',
    'description': 'Get the weather for a given location.',
    'approval_mode': 'never_require',
    'parameters': [   {   'name': 'location',
                          'type': [<class 'str'>],
                          'required': True,
                          'default': None},
                      {   'name': 'ctx',
                          'type': [   <class 'agent_framework._middleware.FunctionInvocationContext'>],
                          'required': False,
                          'default': None},
                      {   'name': 'city',
                          'type': [<class 'str'>],
                          'required': False,
                          'default': 'Halifax'}],
    'return_type': [<class 'str'>]}


## Inspect Class-Based Tool Decorator

```python
class WeatherTools:
    """Tools that share a service client, feature flags, and a cache.

    These values live They are never in the tool schema,
    so the model can't see or set them.
    """

    def __init__(self, service: FakeWeatherService, *, units: str = "celsius",
                 enable_forecast: bool = True) -> None:
        self._service = service                  # shared dependency
        self._units = units                      # config the model can't override
        self._enable_forecast = enable_forecast  # feature flag
        self._cache: dict[str, dict] = {}        # long-lived mutable state

    async def get_current_weather(
        self,
        city: Annotated[str, "The city to get current weather for"],
    ) -> str:
        """Get the current weather for a city."""
        if city not in self._cache:
            self._cache[city] = await self._service.current(city)
        data = self._cache[city]
        temp = data["temp_c"] if self._units == "celsius" else data["temp_c"] * 9 / 5 + 32
        unit = "°C" if self._units == "celsius" else "°F"
        return f"{city}: {data['conditions']}, {temp:.0f}{unit}"

    async def get_forecast(
        self,
        city: Annotated[str, "The city to forecast"],
        days: Annotated[int, "Number of days (1-5)"] = 3,
    ) -> str:
        """Get a multi-day forecast for a city."""
        if not self._enable_forecast:
            return "Forecasts are currently disabled."
        days = max(1, min(days, 5))
        rows = await self._service.forecast(city, days)
        return "\n".join(f"Day {r['day']}: {r['conditions']}, {r['temp_c']}°C" for r in rows)

    def clear_cache(self) -> str:
        """Clear cached weather results."""
        count = len(self._cache)
        self._cache.clear()
        return f"Cleared {count} cached entries."


async def main() -> None:
    weather = WeatherTools(FakeWeatherService(), units="celsius", enable_forecast=True)

    agent = Agent(
        client=FoundryChatClient(credential=AzureCliCredential()),
        name="WeatherAgent",
        instructions="You are a helpful weather assistant.",
        tools=[
            weather.get_current_weather,  # bound method: `self` is already filled in
            weather.get_forecast,
            # Wrap a bound method with tool() to set options like approval_mode
            tool(weather.clear_cache, approval_mode="always_require"),
        ],
    )

    result = await agent.run("What's the weather in Halifax, and the 3-day forecast?")
    print(result.text)


if __name__ == "__main__":
    asyncio.run(main())
```

LEFT-OFF: https://codehalwell.github.io/AgentGuides/microsoft-agent-framework-guide/python/microsoft_agent_framework_python_tools/

```python
 @tool(
     name="weather_tool",
     description="Returns current conditions for a city.",
     approval_mode="never_require",   # or "always_require"
     max_invocations=50,              # lifetime cap on this tool instance
     max_invocation_exceptions=3,     # stop calling it after 3 failures
     kind="readonly",                 # free-form label (see "Tool kind" below)
     additional_properties={"team": "platform"},
)
```

Does schema make a difference, or does the framework parse it anyway?

```python
class WeatherInput(BaseModel):
    location: Annotated[str, Field(description="City name")]
    unit: Literal["celsius", "fahrenheit"] = "celsius"


@tool(schema=WeatherInput)
def get_weather(location: str, unit: str = "celsius") -> str:
    """Get the weather."""
    return f"{location}: 22 {unit}"
```

kind is an optional classification string, and additional_properties a free-form dict. The framework does not interpret either, with one exception: kind="shell" marks a shell tool, which the OpenAI and Anthropic clients send as their native shell tool (see client.get_shell_tool(...)). Any other value is yours to use, typically in function middleware:

## Inspect MCP Tool

## Inspect Agent Function Invocation Configuration

Set limits on the chat client to control automatic tool invocation by model round trips, total function calls, and elapsed wall-clock time

## Inspect Agent as Tool

Scenarios to test:

- @tool()
- @tool(approval_mode, name, description)
- @tool(schema)
- class-based (tool decorator applied to a method)
- class-based (no tool decorator applied)

In [91]:
# Tool without approval mode
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str | int, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str | int:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [52]:
print(type(get_weather))

<class 'agent_framework._tools.FunctionTool'>


In [58]:
dir(get_weather)

['DEFAULT_EXCLUDE',
 'INJECTABLE',
 '_SHALLOW_COPY_FIELDS',
 '__annotations__',
 '__call__',
 '__class__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_cached_parameters',
 '_context_parameter_name',
 '_discover_injected_parameters',
 '_get_type_identifier',
 '_input_model_explicitly_provided',
 '_input_schema',
 '_input_schema_cached',
 '_instance',
 '_invocation_duration_histogram',
 '_is_context_parameter',
 '_make_dumpable',
 '_resolve_input_model',
 '_schema_supplied',
 'additional_properties',
 'approval_mode',
 'declaration_only',
 'description',
 'from_dict',
 '

In [ ]:
# Tool without approval mode
@tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [46]:
class WeatherTool(BaseModel):
    location: Annotated[str, Field(description="The location to get the weather for.")]

# Tool without approval mode
@tool(schema=WeatherTool)
def get_weather_schema(
    location: str,
    ctx: FunctionInvocationContext,
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [ ]:
class MyToolClassWithHandlers:
    @tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
    def get_weather_method(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")],
        ctx: FunctionInvocationContext,
    ) -> str:
        """Get the weather for a given location."""
        return f"The weather in {location} is cloudy with a high of 15°C."


<class 'agent_framework._tools.FunctionTool'>


In [ ]:
class MyToolClassWithoutHandlers:
    def divide(
        self,
        a: Annotated[float, Field(description="The dividend")],
        b: Annotated[float, Field(description="The divisor")],
        ctx: FunctionInvocationContext
    ) -> float:
        """Divide two numbers."""
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

    def add(
        self,
        a: Annotated[float, Field(description="The first addend")],
        b: Annotated[float, Field(description="The second addend")],
        ctx: FunctionInvocationContext
    ) -> float:
        """Add two numbers."""
        return a + b

In [49]:
inspect.signature(get_weather)

ValueError: no signature found for builtin <agent_framework._tools.FunctionTool object at 0x0000024FBB90C190>

In [ ]:
get_weather.approval_mode

'never_require'

In [ ]:
get_weather.description

'Get the weather for a given location.'

In [92]:
get_weather.parameters()

{'properties': {'location': {'anyOf': [{'type': 'string'},
    {'type': 'integer'}],
   'description': 'The location to get the weather for.',
   'title': 'Location'}},
 'required': ['location'],
 'title': 'get_weather_input',
 'type': 'object'}

In [ ]:
inspect.getdoc(get_weather.func)

'Get the weather for a given location.'

In [59]:
dir(inspect.signature(get_weather.func))

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '_bind',
 '_bound_arguments_cls',
 '_hash_basis',
 '_parameter_cls',
 '_parameters',
 '_return_annotation',
 'bind',
 'bind_partial',
 'empty',
 'format',
 'from_callable',
 'parameters',
 'replace',
 'return_annotation']

In [86]:
inspect.signature(get_weather.func).return_annotation

str | int

In [93]:
get_weather.to_json()

'{"type": "function_tool", "name": "get_weather", "description": "Get the weather for a given location.", "approval_mode": "never_require", "invocation_count": 0, "invocation_exception_count": 0, "input_model": {"properties": {"location": {"anyOf": [{"type": "string"}, {"type": "integer"}], "description": "The location to get the weather for.", "title": "Location"}}, "required": ["location"], "title": "get_weather_input", "type": "object"}}'

In [87]:
# This is how we can get the actual types, not just a string representation 
import typing
import types

hints = typing.get_type_hints(get_weather.func)
return_type = hints.get("return", None)

origin = typing.get_origin(return_type)
args = typing.get_args(return_type)

if origin in (typing.Union, types.UnionType):   # handles both forms
    print(args)
else:
    print("Not a union:", return_type)


{'location': <class 'str'>, 'ctx': <class 'agent_framework._middleware.FunctionInvocationContext'>, 'return': str | int}
(<class 'str'>, <class 'int'>)


In [94]:
# LEFT-OFF: if Gemini suggests to use the string based version, then can we do the same with executors?